# FastAPI

### 1.3 모델 배포에 FastAPI가 적합한 이유

아래 코드를 직접 실행해 보세요.
Pydantic은 검증 실패 시 ValidationError를 발생시키며,
어떤 필드가, 왜 잘못되었는지 상세하게 알려줍니다.
FastAPI에서는 이 검증이 요청 수신 시 자동으로 수행되어 422 에러로 반환됩니다.

In [1]:
# 입력 스키마를 클래스로 선언합니다
from pydantic import BaseModel, Field, ValidationError

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1)    # 빈 문자열 불가

# FastAPI가 자동으로 처리하는 것들:
# - text 필드가 없으면 → 422 에러 + "field required" 메시지
# - text가 문자열이 아니면 → 422 에러 + "string type expected" 메시지
# - text가 빈 문자열이면 → 422 에러 + "min_length" 메시지

In [2]:
# 정상 입력 — 통과
req = PredictRequest(text="이 영화 재밌다")
print(f"✅ 정상: {req.text}")

✅ 정상: 이 영화 재밌다


In [3]:
# 에러 1: text 필드 누락
try:
    PredictRequest()
except ValidationError as e:
    print(f"\n❌ 필드 누락:\n{e}")
# Field required


❌ 필드 누락:
1 validation error for PredictRequest
text
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


In [4]:
# 에러 2: 잘못된 타입 (문자열이어야 하는데 리스트를 전달)
try:
    PredictRequest(text=["이것은", "리스트"])
except ValidationError as e:
    print(f"\n❌ 타입 오류:\n{e}")
# Input should be a valid string


❌ 타입 오류:
1 validation error for PredictRequest
text
  Input should be a valid string [type=string_type, input_value=['이것은', '리스트'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


In [5]:
# 에러 3: 빈 문자열 (min_length=1 위반)
try:
    PredictRequest(text="")
except ValidationError as e:
    print(f"\n❌ 빈 문자열:\n{e}")
# String should have at least 1 character


❌ 빈 문자열:
1 validation error for PredictRequest
text
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short


### 서버 코드 작성

In [1]:
%%writefile app/main_basic.py
"""
최소한의 FastAPI 서버
"""
from fastapi import FastAPI

# FastAPI 인스턴스 생성
app = FastAPI(
    title="My First ML API",
    description="Day 2 실습: 첫 번째 FastAPI 서버",
    version="0.1.0",
)

# 엔드포인트 1: 헬스체크 (서버가 살아있는지 확인)
@app.get("/health")
def health_check():
    return {"status": "healthy"}

# 엔드포인트 2: 루트 경로
@app.get("/")
def root():
    return {
        "message": "ML Model Serving API",
        "docs_url": "/docs",
    }

Overwriting app/main_basic.py


### 주피터 노트북에서 실행하는 방법

In [3]:
# 노트북 환경에서 uvicorn을 실행하기 위한 설정
!pip install nest_asyncio -q

import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

# 백그라운드 스레드에서 서버 실행
def run_server():
    uvicorn.run("app.main_basic:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)   # 서버가 시작될 때까지 잠시 대기
print("✅ 서버가 시작되었습니다: http://localhost:8000")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
INFO:     Started server process [19768]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버가 시작되었습니다: http://localhost:8000


### 서버 호출 테스트

In [8]:
import requests

# 헬스체크 엔드포인트 호출
response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

INFO:     127.0.0.1:1427 - "GET /health HTTP/1.1" 200 OK
상태 코드: 200
응답: {'status': 'healthy'}


In [9]:
# 루트 엔드포인트 호출
response = requests.get("http://localhost:8000/")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

INFO:     127.0.0.1:1467 - "GET / HTTP/1.1" 200 OK
상태 코드: 200
응답: {'message': 'ML Model Serving API', 'docs_url': '/docs'}


## 2. 첫 번째 엔드포인트 만들기: Path, Query, Body

### 2. 첫 번째 엔드포인트 만들기: Path, Query, Body

In [1]:
%%writefile app/main_params.py
"""
파라미터 방식 실습
"""
from fastapi import FastAPI

app = FastAPI(title="Parameter Examples")

# ===== Path 파라미터 =====

# 기본 사용: 중괄호 {}로 경로 변수를 선언합니다
@app.get("/models/{model_name}")
def get_model_info(model_name: str):
    """특정 모델의 정보를 반환합니다."""
    return {
        "model_name": model_name,
        "status": "running",
        "version": "1.0.0",
    }

Overwriting app/main_params.py


In [1]:
# 서버 실행 (이전 서버가 실행 중이면 커널 재시작 후 실행)
import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_params:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

INFO:     Started server process [26716]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨
INFO:     127.0.0.1:4006 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:4006 - "GET /openapi.json HTTP/1.1" 200 OK


In [3]:
import requests

# Path 파라미터 테스트
response = requests.get("http://localhost:8000/models/sentiment-v1")
print(response.json())
# {'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}

response = requests.get("http://localhost:8000/models/image-classifier")
print(response.json())
# {'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}

INFO:     127.0.0.1:7081 - "GET /models/sentiment-v1 HTTP/1.1" 200 OK
{'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}
INFO:     127.0.0.1:1467 - "GET /models/image-classifier HTTP/1.1" 200 OK
{'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}


### 타입 지정의 효과

In [3]:
%%writefile -a app/main_params.py

# Path 파라미터에 int 타입 지정
@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    """특정 예측 결과를 조회합니다."""
    return {
        "prediction_id": prediction_id,
        "label": "긍정",
        "confidence": 0.92,
    }

Appending to app/main_params.py


In [4]:
# 정상 요청: 숫자를 전달
response = requests.get("http://localhost:8000/predictions/42")
print(f"상태: {response.status_code}, 응답: {response.json()}")
# 상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}

# 잘못된 요청: 문자열을 전달
response = requests.get("http://localhost:8000/predictions/abc")
print(f"상태: {response.status_code}")
print(f"에러: {response.json()}")
# 상태: 422
# 에러: {'detail': [{'type': 'int_parsing', 'msg': 'Input should be a valid integer...'}]}

INFO:     127.0.0.1:1932 - "GET /predictions/42 HTTP/1.1" 200 OK
상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}
INFO:     127.0.0.1:1936 - "GET /predictions/abc HTTP/1.1" 422 Unprocessable Entity
상태: 422
에러: {'detail': [{'type': 'int_parsing', 'loc': ['path', 'prediction_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


### 2.3 Query 파라미터 — URL 뒤에 조건을 추가

In [5]:
%%writefile -a app/main_params.py

# ===== Query 파라미터 =====

# 함수 인자 중 Path에 포함되지 않은 것은 자동으로 Query 파라미터가 됩니다
@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    """
    모델 목록을 조회합니다.

    - status: 필터링 조건 (선택) — "running", "stopped" 등
    - limit: 반환할 최대 개수 (기본값: 10)
    """
    # 실제로는 DB에서 조회하겠지만, 여기서는 예시 데이터를 반환합니다
    models = [
        {"name": "sentiment-v1", "status": "running"},
        {"name": "image-clf-v2", "status": "running"},
        {"name": "ner-v1", "status": "stopped"},
    ]

    # status 필터링
    if status:
        models = [m for m in models if m["status"] == status]

    # limit 적용
    models = models[:limit]

    return {
        "total": len(models),
        "models": models,
    }

Appending to app/main_params.py


In [3]:
# 파라미터 없이 호출 (기본값 사용)
response = requests.get("http://localhost:8000/models")
print("전체 모델:", response.json())

# status로 필터링
response = requests.get("http://localhost:8000/models?status=running")
print("running만:", response.json())

# 여러 파라미터 조합
response = requests.get("http://localhost:8000/models?status=running&limit=1")
print("running, 1개만:", response.json())

INFO:     127.0.0.1:11428 - "GET /models HTTP/1.1" 200 OK
전체 모델: {'total': 3, 'models': [{'name': 'sentiment-v1', 'status': 'running'}, {'name': 'image-clf-v2', 'status': 'running'}, {'name': 'ner-v1', 'status': 'stopped'}]}
INFO:     127.0.0.1:11430 - "GET /models?status=running HTTP/1.1" 200 OK
running만: {'total': 2, 'models': [{'name': 'sentiment-v1', 'status': 'running'}, {'name': 'image-clf-v2', 'status': 'running'}]}
INFO:     127.0.0.1:11434 - "GET /models?status=running&limit=1 HTTP/1.1" 200 OK
running, 1개만: {'total': 1, 'models': [{'name': 'sentiment-v1', 'status': 'running'}]}


### 2.4 Request Body — 본문에 JSON 데이터 전달

In [2]:
%%writefile -a app/main_params.py

# ===== Request Body =====
from pydantic import BaseModel, Field
from typing import Optional

# 입력 스키마 정의
# class PredictRequest(BaseModel):
#     text: str
#     return_probabilities: bool = False    # 선택, 기본값 False
# Field()에 description과 examples를 추가하면 Swagger UI에 반영됩니다

class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분석할 텍스트. 1자 이상 5000자 이하.",
        examples=["이 영화 정말 재밌다"],
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 각 클래스별 확률을 함께 반환합니다.",
    )

# 출력 스키마 정의
class PredictResponse(BaseModel):
    label: str
    confidence: float
    probabilities: Optional[dict] = None

    
@app.post("/predict", response_model=PredictResponse, summary="텍스트 감성 분석")
def predict(request: PredictRequest):
    """
    텍스트 감성 분석을 수행합니다.

    - text: 분석할 텍스트 (필수)
    - return_probabilities: 전체 확률을 반환할지 여부 (선택, 기본 False)
    """
    # 실제로는 모델 추론을 수행하겠지만, 여기서는 더미 결과를 반환합니다
    result = {
        "label": "긍정",
        "confidence": 0.92,
    }

    if request.return_probabilities:
        result["probabilities"] = {
            "긍정": 0.92,
            "부정": 0.05,
            "중립": 0.03,
        }

    return result

Appending to app/main_params.py


In [3]:
# POST 요청: JSON 데이터를 본문에 담아 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "이 영화 정말 재밌다"}
)
print("기본 응답:", response.json())

# 옵션 추가
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "text": "이 영화 정말 재밌다",
        "return_probabilities": True
    }
)
print("확률 포함:", response.json())

INFO:     127.0.0.1:6999 - "POST /predict HTTP/1.1" 200 OK
기본 응답: {'label': '긍정', 'confidence': 0.92, 'probabilities': None}
INFO:     127.0.0.1:7002 - "POST /predict HTTP/1.1" 200 OK
확률 포함: {'label': '긍정', 'confidence': 0.92, 'probabilities': {'긍정': 0.92, '부정': 0.05, '중립': 0.03}}


In [4]:
# text 필드 누락
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Field required

# text에 잘못된 타입 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": 12345}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Input should be a valid string

INFO:     127.0.0.1:7057 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태: 422
에러: Field required
INFO:     127.0.0.1:7060 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태: 422
에러: Input should be a valid string
INFO:     127.0.0.1:10728 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:10728 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:11223 - "GET /models/sentiment-v1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:7929 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:3210 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


## 3. Swagger UI로 API 테스트하기

In [5]:
# FastAPI가 자동 생성한 OpenAPI 스펙 확인
import requests, json

response = requests.get("http://localhost:8000/openapi.json")
spec = response.json()

print(f"API 제목: {spec['info']['title']}")
print(f"API 버전: {spec['info']['version']}")
print(f"\n등록된 엔드포인트:")
for path, methods in spec['paths'].items():
    for method in methods:
        print(f"  {method.upper():6s} {path}")

INFO:     127.0.0.1:12820 - "GET /openapi.json HTTP/1.1" 200 OK
API 제목: Parameter Examples
API 버전: 0.1.0

등록된 엔드포인트:
  GET    /models/{model_name}
  GET    /predictions/{prediction_id}
  GET    /models
  POST   /predict


In [6]:
# PredictRequest의 JSON Schema 확인
predict_schema = spec['components']['schemas']['PredictRequest']
print("PredictRequest 스키마:")
print(json.dumps(predict_schema, indent=2, ensure_ascii=False))

PredictRequest 스키마:
{
  "properties": {
    "text": {
      "type": "string",
      "title": "Text"
    },
    "return_probabilities": {
      "type": "boolean",
      "title": "Return Probabilities",
      "default": false
    }
  },
  "type": "object",
  "required": [
    "text"
  ],
  "title": "PredictRequest"
}


### 3.5 문서 품질 높이기: 알아두면 유용한 옵션들

In [7]:
from pydantic import BaseModel, Field

# Field()에 description과 examples를 추가하면 Swagger UI에 반영됩니다
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분석할 텍스트. 1자 이상 5000자 이하.",
        examples=["이 영화 정말 재밌다"],
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 각 클래스별 확률을 함께 반환합니다.",
    )


In [8]:
# 엔드포인트에 summary를 추가하면 Swagger UI에서 짧은 제목으로 표시됩니다
@app.post("/predict", summary="텍스트 감성 분석")
def predict(request: PredictRequest):
    """입력된 텍스트의 감성을 분석합니다."""
    ...

NameError: name 'app' is not defined

INFO:     127.0.0.1:11161 - "GET /redoc HTTP/1.1" 200 OK
INFO:     127.0.0.1:5373 - "GET /redoc HTTP/1.1" 200 OK
INFO:     127.0.0.1:2528 - "GET /redocs HTTP/1.1" 404 Not Found


### 3.6 ReDoc — 또 다른 자동 문서

## 4. Pydantic을 활용한 입력 데이터 검증(Schema)

In [2]:
from pydantic import BaseModel
from typing import Optional

# Pydantic 모델 정의 = 데이터의 "설계도"
class PredictRequest(BaseModel):
    text: str                              # 필수, 문자열
    language: str = "ko"                   # 선택, 기본값 "ko"
    return_probabilities: bool = False     # 선택, 기본값 False
    top_k: Optional[int] = None            # 선택, 없으면 None

In [3]:
# 정상적인 데이터로 인스턴스 생성
req = PredictRequest(text="이 영화 재밌다")
print(f"text: {req.text}")
print(f"language: {req.language}")               # 기본값 "ko"
print(f"return_probabilities: {req.return_probabilities}")  # 기본값 False
print(f"top_k: {req.top_k}")                     # 기본값 None

text: 이 영화 재밌다
language: ko
return_probabilities: False
top_k: None


In [4]:
# 자동 타입 변환
req2 = PredictRequest(text="테스트", top_k="3")   # 문자열 "3"이 int 3으로 변환
print(f"top_k: {req2.top_k}, 타입: {type(req2.top_k)}")
# top_k: 3, 타입:

top_k: 3, 타입: <class 'int'>


In [5]:
# 검증 실패: 필수 필드 누락
from pydantic import ValidationError

try:
    req3 = PredictRequest()   # text가 없음
except ValidationError as e:
    print("검증 실패!")
    print(e)

검증 실패!
1 validation error for PredictRequest
text
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


### 4.3 Field를 활용한 상세 검증

In [6]:
from pydantic import BaseModel, Field

class TextInput(BaseModel):
    text: str = Field(
        ...,                        # ... 은 "필수"를 의미합니다
        min_length=1,               # 최소 1글자 (빈 문자열 방지)
        max_length=5000,            # 최대 5000자 (비정상 입력 방지)
        description="분석할 텍스트",
    )


In [7]:
# 테스트
try:
    TextInput(text="")   # 빈 문자열
except ValidationError as e:
    print(f"빈 문자열: {e.errors()[0]['msg']}")
# 빈 문자열: String should have at least 1 character

try:
    TextInput(text="a" * 5001)   # 5000자 초과
except ValidationError as e:
    print(f"초과: {e.errors()[0]['msg']}")
# 초과: String should have at most 5000 characters

빈 문자열: String should have at least 1 character
초과: String should have at most 5000 characters


#### 숫자 검증

In [8]:
class InferenceOptions(BaseModel):
    temperature: float = Field(
        default=1.0,
        gt=0.0,       # greater than: 0보다 커야 함
        le=2.0,        # less than or equal: 2 이하
        description="생성 온도. 0 초과, 2 이하.",
    )
    top_k: int = Field(
        default=5,
        ge=1,          # greater than or equal: 1 이상
        le=100,        # less than or equal: 100 이하
        description="반환할 상위 결과 수",
    )
    batch_size: int = Field(
        default=1,
        ge=1,
        le=32,
        description="배치 크기. 1 이상 32 이하.",
    )

In [9]:
# 정상
opts = InferenceOptions(temperature=0.7, top_k=10)
print(f"temperature: {opts.temperature}, top_k: {opts.top_k}")

# 범위 초과
try:
    InferenceOptions(temperature=0.0)   # gt=0.0 이므로 0은 불가
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be greater than 0

try:
    InferenceOptions(top_k=0)   # ge=1 이므로 0은 불가
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be greater than or equal to 1

temperature: 0.7, top_k: 10
에러: Input should be greater than 0
에러: Input should be greater than or equal to 1


#### 선택지 제한: Literal

In [10]:
from typing import Literal

class AnalysisRequest(BaseModel):
    text: str = Field(..., min_length=1)
    language: Literal["ko", "en", "ja"] = Field(
        default="ko",
        description="지원 언어: ko(한국어), en(영어), ja(일본어)",
    )
    task: Literal["sentiment", "summary", "ner"] = Field(
        ...,
        description="수행할 작업 유형",
    )



In [11]:
# 정상
req = AnalysisRequest(text="테스트", task="sentiment")
print(f"language: {req.language}, task: {req.task}")

# 허용되지 않은 값
try:
    AnalysisRequest(text="test", language="fr", task="sentiment")
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be 'ko', 'en' or 'ja'

language: ko, task: sentiment
에러: Input should be 'ko', 'en' or 'ja'


#### 4.4 응답 스키마와 response_model

In [5]:
from pydantic import BaseModel, Field

class PredictResponse(BaseModel):
    label: str = Field(description="예측 레이블")
    confidence: float = Field(description="확신도 (0.0 ~ 1.0)", ge=0.0, le=1.0)

In [6]:
@app.post("/predict", response_model=PredictResponse)
def predict(request: TextInput):
    # 추론 로직...
    return PredictResponse(label="긍정", confidence=0.92)

NameError: name 'app' is not defined

#### 4.5 422 에러 응답의 구조

In [2]:
# 의도적으로 잘못된 요청을 보내서 에러 구조를 확인합니다
import requests, json

response = requests.post(
    "http://localhost:8000/predict",
    json={
        "return_probabilities": "yes"   # bool이어야 하는데 문자열
        # text 필드 누락
    }
)

print(f"상태 코드: {response.status_code}")
print(f"에러 응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

INFO:     127.0.0.1:8635 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태 코드: 422
에러 응답:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "text"
      ],
      "msg": "Field required",
      "input": {
        "return_probabilities": "yes"
      }
    }
  ]
}


## 5. 실습: 모델 추론 엔드포인트 구현 및 테스트

### Step 1 — model_utils.py 확인 및 보완

In [1]:
%%writefile app/model_utils.py
"""
모델 로드 및 추론 유틸리티
FastAPI 엔드포인트가 이 모듈을 import하여 사용합니다.
"""

import torch
import torch.nn as nn
from torchvision import transforms


# ===== 모델 정의 =====
class SimpleClassifier(nn.Module):
    """
    간단한 이미지 분류 모델
    - 입력: 1x28x28 (MNIST와 동일한 크기)
    - 출력: 10개 클래스에 대한 확률
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ===== 전처리 파이프라인 =====
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


# ===== 모델 로드 =====
def load_model(model_path: str = "models/mnist_state_dict.pth") -> nn.Module:
    """
    저장된 state_dict를 로드하여 추론 가능한 모델을 반환합니다.
    """
    model = SimpleClassifier(num_classes=10)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    return model


# ===== 추론 함수 =====
def predict(model: nn.Module, input_tensor: torch.Tensor) -> dict:
    """
    모델에 입력 텐서를 전달하고 예측 결과를 반환합니다.

    Args:
        model: 로드된 PyTorch 모델
        input_tensor: 전처리된 입력 텐서 (1, 1, 28, 28)

    Returns:
        dict: {"label": int, "confidence": float, "probabilities": list}
    """
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, dim=1)

    return {
        "label": predicted.item(),
        "confidence": round(confidence.item(), 4),
        "probabilities": probabilities[0].tolist(),
    }

Overwriting app/model_utils.py


In [2]:
from app.model_utils import load_model, predict, preprocess
print("✅ model_utils import 성공")

# 모델 로드 테스트
model = load_model("models/mnist_state_dict.pth")
print(f"✅ 모델 로드 성공: {type(model).__name__}")

✅ model_utils import 성공
✅ 모델 로드 성공: SimpleClassifier


### 5.4 Step 2 — Pydantic 스키마 설계
- 스키마를 별도 파일로 분리합니다

In [4]:
%%writefile app/schemas.py
"""
API 입출력 스키마 정의
"""

from pydantic import BaseModel, Field
from typing import Optional


class PredictRequest(BaseModel):
    """모델 추론 요청 스키마"""
    pixel_values: list[float] = Field(
        ...,
        min_length=784,       # 28 * 28 = 784
        max_length=784,
        description="28x28 이미지의 픽셀 값 (784개). 0.0~1.0 범위.",
        examples=[[0.0] * 784],   # Swagger UI에 예시로 표시
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 전체 클래스별 확률을 함께 반환합니다.",
    )


class PredictResponse(BaseModel):
    """모델 추론 응답 스키마"""
    label: int = Field(
        description="예측된 숫자 (0~9)",
    )
    confidence: float = Field(
        description="예측 확신도 (0.0~1.0)",
    )
    probabilities: Optional[list[float]] = Field(
        default=None,
        description="각 클래스(0~9)별 확률. return_probabilities=True일 때만 포함.",
    )
    model_version: str = Field(
        default="1.0.0",
        description="사용된 모델 버전",
    )


class HealthResponse(BaseModel):
    """헬스체크 응답 스키마"""
    status: str
    model_loaded: bool

Writing app/schemas.py


### 5.5 Step 3 — FastAPI 서버 작성

In [5]:
%%writefile app/main.py
"""
Day 2 실습: 모델 추론 API 서버
"""

from fastapi import FastAPI, HTTPException
import torch

from app.model_utils import load_model, predict
from app.schemas import (
    PredictRequest,
    PredictResponse,
    HealthResponse,
)

# ===== FastAPI 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API",
    description="Day 2 실습: MNIST 숫자 분류 모델 추론 API",
    version="1.0.0",
)


# ===== 모델을 서버 시작 시 한 번만 로드 =====
# 모듈 레벨에서 로드하면 서버가 시작될 때 실행됩니다.
# 요청마다 로드하면 매번 수 초가 걸리므로, 반드시 한 번만 로드해야 합니다.
try:
    model = load_model("models/mnist_state_dict.pth")
    model_loaded = True
    print("✅ 모델 로드 완료")
except Exception as e:
    model = None
    model_loaded = False
    print(f"❌ 모델 로드 실패: {e}")


# ===== 엔드포인트 1: 헬스체크 =====
@app.get("/health", response_model=HealthResponse)
def health_check():
    """서버 상태와 모델 로드 여부를 확인합니다."""
    return HealthResponse(
        status="healthy",
        model_loaded=model_loaded,
    )


# ===== 엔드포인트 2: 모델 추론 =====
@app.post("/predict", response_model=PredictResponse, summary="MNIST 숫자 예측")
def predict_digit(request: PredictRequest):
    """
    28x28 이미지의 픽셀 값을 받아 숫자(0~9)를 예측합니다.

    - **pixel_values**: 784개의 float 리스트 (28x28 이미지)
    - **return_probabilities**: True로 설정하면 전체 확률 분포를 반환
    """
    # 1. 모델이 로드되었는지 확인
    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="모델이 로드되지 않았습니다. 서버 로그를 확인하세요."
        )

    # 2. 입력 데이터를 텐서로 변환
    try:
        input_tensor = torch.tensor(request.pixel_values, dtype=torch.float32)
        input_tensor = input_tensor.reshape(1, 1, 28, 28)  # (batch, channel, H, W)
    except Exception as e:
        raise HTTPException(
            status_code=400,
            detail=f"입력 데이터를 텐서로 변환할 수 없습니다: {str(e)}"
        )

    # 3. 추론 실행
    try:
        result = predict(model, input_tensor)
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"모델 추론 중 에러가 발생했습니다: {str(e)}"
        )

    # 4. 응답 생성
    response = PredictResponse(
        label=result["label"],
        confidence=result["confidence"],
        model_version="1.0.0",
    )

    # 5. 옵션: 확률 분포 포함
    if request.return_probabilities:
        response.probabilities = [round(p, 4) for p in result["probabilities"]]

    return response

Writing app/main.py


### 5.6 Step 4 — 서버 실행 및 테스트

#### 서버 실행

In [6]:
# 노트북 환경에서 실행
import nest_asyncio, uvicorn, threading, time

nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # 모델 로드 시간 고려하여 3초 대기
print("✅ 서버 시작됨: http://localhost:8000")

INFO:     Started server process [24468]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 모델 로드 완료
✅ 서버 시작됨: http://localhost:8000
INFO:     127.0.0.1:5602 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:5602 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:11221 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:11221 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:13076 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:13076 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:13076 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:13076 - "GET /openapi.json HTTP/1.1" 200 OK


#### 테스트 1: 헬스체크

In [7]:
import requests

response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

INFO:     127.0.0.1:2408 - "GET /health HTTP/1.1" 200 OK
상태 코드: 200
응답: {'status': 'healthy', 'model_loaded': True}


### 테스트 2: 실제 MNIST 이미지로 추론

In [8]:
from torchvision import datasets, transforms

# MNIST 테스트 데이터 로드
test_dataset = datasets.MNIST(
    root="data", train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
)

# 첫 번째 테스트 이미지 가져오기
test_image, true_label = test_dataset[0]
print(f"이미지 크기: {test_image.shape}")     # torch.Size([1, 28, 28])
print(f"정답 레이블: {true_label}")

# 픽셀 값을 리스트로 변환 (API에 보낼 형식)
pixel_values = test_image.flatten().tolist()
print(f"픽셀 값 개수: {len(pixel_values)}")   # 784

이미지 크기: torch.Size([1, 28, 28])
정답 레이블: 7
픽셀 값 개수: 784


In [9]:
import json

# 추론 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": False,
    }
)

print(f"상태 코드: {response.status_code}")
print(f"응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))



INFO:     127.0.0.1:7355 - "POST /predict HTTP/1.1" 200 OK
상태 코드: 200
응답:
{
  "label": 7,
  "confidence": 1.0,
  "probabilities": null,
  "model_version": "1.0.0"
}


### 테스트 3: 확률 분포 포함 요청

In [10]:
# return_probabilities를 True로 설정
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": True,
    }
)

result = response.json()
print(f"예측: {result['label']} (확신도: {result['confidence']})")
print(f"\n클래스별 확률:")
for i, prob in enumerate(result['probabilities']):
    bar = "█" * int(prob * 50)
    print(f"  {i}: {prob:.4f} {bar}")

INFO:     127.0.0.1:7458 - "POST /predict HTTP/1.1" 200 OK
예측: 7 (확신도: 1.0)

클래스별 확률:
  0: 0.0000 
  1: 0.0000 
  2: 0.0000 
  3: 0.0000 
  4: 0.0000 
  5: 0.0000 
  6: 0.0000 
  7: 1.0000 ██████████████████████████████████████████████████
  8: 0.0000 
  9: 0.0000 


### 테스트 4: 여러 이미지 연속 테스트

In [11]:
# 10개 이미지를 연속으로 테스트
print(f"{'이미지':<8} {'정답':<6} {'예측':<6} {'확신도':<10} {'결과'}")
print("-" * 45)

correct = 0
for i in range(10):
    image, true_label = test_dataset[i]
    pixel_values = image.flatten().tolist()

    response = requests.post(
        "http://localhost:8000/predict",
        json={"pixel_values": pixel_values}
    )
    result = response.json()

    is_correct = result["label"] == true_label
    if is_correct:
        correct += 1

    mark = "✅" if is_correct else "❌"
    print(f"  #{i:<5} {true_label:<6} {result['label']:<6} {result['confidence']:<10} {mark}")

print(f"\n정확도: {correct}/10 ({correct * 10}%)")

이미지      정답     예측     확신도        결과
---------------------------------------------
INFO:     127.0.0.1:7600 - "POST /predict HTTP/1.1" 200 OK
  #0     7      7      1.0        ✅
INFO:     127.0.0.1:7603 - "POST /predict HTTP/1.1" 200 OK
  #1     2      2      1.0        ✅
INFO:     127.0.0.1:7606 - "POST /predict HTTP/1.1" 200 OK
  #2     1      1      0.9999     ✅
INFO:     127.0.0.1:7609 - "POST /predict HTTP/1.1" 200 OK
  #3     0      0      1.0        ✅
INFO:     127.0.0.1:7613 - "POST /predict HTTP/1.1" 200 OK
  #4     4      4      1.0        ✅
INFO:     127.0.0.1:7615 - "POST /predict HTTP/1.1" 200 OK
  #5     1      1      1.0        ✅
INFO:     127.0.0.1:7619 - "POST /predict HTTP/1.1" 200 OK
  #6     4      4      0.9996     ✅
INFO:     127.0.0.1:7623 - "POST /predict HTTP/1.1" 200 OK
  #7     9      9      1.0        ✅
INFO:     127.0.0.1:7625 - "POST /predict HTTP/1.1" 200 OK
  #8     5      5      0.9991     ✅
INFO:     127.0.0.1:7629 - "POST /predict HTTP/1.1" 200 OK
  #

### 5.7 Step 5 — 에러 상황 테스트

In [12]:
# 784개가 아닌 100개만 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": [0.0] * 100}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러 메시지: {response.json()['detail'][0]['msg']}")

INFO:     127.0.0.1:8309 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태 코드: 422
에러 메시지: List should have at least 784 items after validation, not 100


In [13]:
# 숫자가 아닌 문자열 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": "이것은 이미지가 아닙니다"}
)
print(f"상태 코드: {response.status_code}")  # 422

INFO:     127.0.0.1:8331 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태 코드: 422


In [14]:
# pixel_values 없이 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러: {response.json()['detail'][0]['msg']}")

INFO:     127.0.0.1:8349 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태 코드: 422
에러: Field required


In [15]:
response = requests.post(
    "http://localhost:8000/predict",
    json={}
)
print(f"상태 코드: {response.status_code}")  # 422

INFO:     127.0.0.1:8368 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
상태 코드: 422
INFO:     127.0.0.1:13672 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:13672 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:11120 - "POST /predict HTTP/1.1" 200 OK
